# Merge 10 TypePro shards and publish the final dataset

Settings: **Internet ON**, accelerator **None/CPU**. Add
`TYPEPRO_FINAL_USERNAME/KEY` for `duyvu1105`. Shards owned
by the final account remain private; shards owned by the second account
are public so this credential can download both groups. Run only after
all `10` shard Datasets have completed successfully.


In [ ]:
SHARD_COUNT = 10
REPOSITORY = 'https://github.com/duyvu1105/TypePro.git'
BRANCH = 'main'
EXPECTED_DATASET_OWNER = 'duyvu1105'
SHARD_SOURCE_CONFIG = [{'label': 'runner_a', 'secret_prefix': 'TYPEPRO_RUNNER_A', 'owner': 'duyvu1105', 'shards': [0, 1, 2, 3, 4], 'public_dataset': False}, {'label': 'runner_b', 'secret_prefix': 'TYPEPRO_RUNNER_B', 'owner': 'duymign', 'shards': [5, 6, 7, 8, 9], 'public_dataset': True}]
FINAL_DATASET_SLUG = "typepro-python-contrastive"
SEED = 13

import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def optional_secret(name):
    try:
        value = secrets.get_secret(name)
    except Exception:
        return None
    value = value.strip() if value else ""
    return value or None

final_username = optional_secret("TYPEPRO_FINAL_USERNAME")
final_key = optional_secret("TYPEPRO_FINAL_KEY")
if not final_username or not final_key:
    raise RuntimeError("Missing TYPEPRO_FINAL_USERNAME/KEY Secrets")
if final_username.casefold() != EXPECTED_DATASET_OWNER.casefold():
    raise RuntimeError(
        f"Final credential belongs to {final_username!r}, expected "
        f"{EXPECTED_DATASET_OWNER!r}"
    )
SHARD_SOURCES = SHARD_SOURCE_CONFIG
FINAL_SOURCE = {
    "label": "final_owner",
    "owner": EXPECTED_DATASET_OWNER,
    "username": final_username,
    "key": final_key,
}
AUTH_ROOT = Path("/kaggle/working/typepro_merge_auth")
AUTH_ROOT.mkdir(parents=True, exist_ok=True)

def use_credential(source):
    auth_config_dir = AUTH_ROOT / source["label"]
    auth_config_dir.mkdir(parents=True, exist_ok=True)
    os.environ["KAGGLE_CONFIG_DIR"] = str(auth_config_dir)
    os.environ["KAGGLE_USERNAME"] = source["username"]
    os.environ["KAGGLE_KEY"] = source["key"]
    os.environ.pop("KAGGLE_API_TOKEN", None)

os.environ["PYTHONUNBUFFERED"] = "1"
print({
    "shard_sources": [
        {"owner": source["owner"], "shards": source["shards"]}
        for source in SHARD_SOURCES
    ],
    "final_dataset_owner": EXPECTED_DATASET_OWNER,
    "credentials_printed": False,
})

REPO_DIR = Path("/kaggle/working/TypePro")
DOWNLOAD_DIR = Path("/kaggle/working/downloaded_shards")
MERGED_BUILD = Path("/kaggle/working/typepro_build")
FINAL_DIR = Path("/kaggle/working/typepro_python_contrastive")

def run(command, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run([str(value) for value in command], cwd=cwd, check=True)


## Clone code and install dependencies


In [ ]:
if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, REPO_DIR])
PIPELINE_DIR = REPO_DIR / "codet5p_type_retrieval"
run([sys.executable, "-m", "pip", "install", "-q", "-U", "-r", PIPELINE_DIR / "requirements-build.txt"])
# Force legacy-key-only authentication for cross-account publishing.
run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "kaggle==1.7.4.2"])


## Download and extract every shard dataset


In [ ]:
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
shard_builds = []
use_credential(FINAL_SOURCE)
for source in SHARD_SOURCES:
    for index in source["shards"]:
        dataset_id = f"{source['owner']}/typepro-build-shard-{index:02d}"
        target = DOWNLOAD_DIR / f"shard_{index:02d}"
        target.mkdir(parents=True, exist_ok=True)
        run(["kaggle", "datasets", "download", "-d", dataset_id, "-p", target, "--unzip"])
        marker_paths = list(target.rglob("shard_manifest.json"))
        if not marker_paths:
            # Older shard datasets may preserve the pre-packed ZIP, while
            # Kaggle expands newer uploads into a directory tree.
            archives = list(target.rglob("typepro_build_shard_*.zip"))
            if len(archives) != 1:
                raise RuntimeError(
                    f"{dataset_id}: expected one build directory or archive, "
                    f"found markers={marker_paths}, archives={archives}"
                )
            with zipfile.ZipFile(archives[0]) as bundle:
                bundle.extractall(target)
            marker_paths = list(target.rglob("shard_manifest.json"))
        if len(marker_paths) != 1:
            raise RuntimeError(f"{dataset_id}: cannot uniquely locate shard build: {marker_paths}")
        build = marker_paths[0].parent
        marker = json.loads(marker_paths[0].read_text(encoding="utf-8"))
        if marker["shard_index"] != index or marker["shard_count"] != SHARD_COUNT or marker["missing_projects"]:
            raise RuntimeError(f"Invalid/incomplete shard marker: {marker}")
        required = [
            build / "metadata" / "split_manifest.json",
            build / "raw_slices",
            build / "project_status",
        ]
        missing = [str(path) for path in required if not path.exists()]
        if missing:
            raise RuntimeError(f"{dataset_id}: missing merge inputs: {missing}")
        shard_builds.append(build)
        print(f"Validated shard {index:02d} from {source['owner']}: {marker['attempted_projects']} projects")
print("All shard build directories:", [str(path) for path in shard_builds])


## Merge shard outputs


In [ ]:
merge_script = PIPELINE_DIR / "merge_shards.py"
run([
    sys.executable, "-u", merge_script,
    "--shard-build-dirs", *shard_builds,
    "--work-dir", MERGED_BUILD,
])


## Finalize contrastive train/validation/test


In [ ]:
prepare = PIPELINE_DIR / "prepare_dataset.py"
run([
    sys.executable, "-u", prepare,
    "--stage", "finalize",
    "--typepro-root", REPO_DIR,
    "--work-dir", MERGED_BUILD,
    "--output-dir", FINAL_DIR,
    "--split-profile", "paper_project",
    "--test-projects", 100,
    "--validation-project-ratio", 0.10,
    "--max-negatives", 7,
    "--seed", SEED,
    "--preview-samples", 2,
    "--preview-max-chars", 1600,
    "--log-every", 10000,
])
run([sys.executable, PIPELINE_DIR / "verify_dataset.py", "--data-dir", FINAL_DIR])


## Display exact counts and examples


In [ ]:
manifest = json.loads((FINAL_DIR / "manifest.json").read_text(encoding="utf-8"))
stats = json.loads((FINAL_DIR / "preprocess_stats.json").read_text(encoding="utf-8"))
recommendation_coverage = {}
for split in ("train", "validation", "test"):
    found = stats.get(f"{split}_gold_recommended", 0)
    missing = stats.get(f"{split}_gold_not_recommended", 0)
    total = found + missing
    recommendation_coverage[split] = {
        "total_samples": total,
        "ground_truth_in_recommendation_types": found,
        "ground_truth_not_in_recommendation_types": missing,
        "percentage": round(100.0 * found / total, 2) if total else 0.0,
    }
print(json.dumps({
    "output": manifest["output"],
    "prepared_counts": manifest["split"]["prepared_counts"],
    "prepared_projects": manifest["split"]["prepared_projects"],
    "preprocess_stats": stats,
    "recommendation_coverage": recommendation_coverage,
}, indent=2, ensure_ascii=False))
print("\n===== GROUND TRUTH IN RECOMMENDATION TYPES =====")
for split, values in recommendation_coverage.items():
    print(
        f"{split}: {values['ground_truth_in_recommendation_types']:,}/"
        f"{values['total_samples']:,} samples ({values['percentage']:.2f}%)"
    )
for split in ("train", "validation", "test"):
    print(f"\n===== {split.upper()} SAMPLES =====")
    with (FINAL_DIR / f"{split}.jsonl").open(encoding="utf-8") as handle:
        for _, line in zip(range(2), handle):
            print(json.dumps(json.loads(line), indent=2, ensure_ascii=False)[:3000])


## Publish final private Kaggle Dataset


In [ ]:
use_credential(FINAL_SOURCE)
final_id = f"{FINAL_SOURCE['owner']}/{FINAL_DATASET_SLUG}"
run([
    sys.executable, PIPELINE_DIR / "publish_kaggle.py",
    "--data-dir", FINAL_DIR,
    "--dataset-id", final_id,
    "--title", "TypePro Python Third-Party Contrastive Data",
    "--message", f"Merge {SHARD_COUNT} verified TypePro shards",
])
completion = {
    "dataset_id": final_id,
    "shard_count": SHARD_COUNT,
    "output": manifest["output"],
}
(FINAL_DIR / "MERGE_COMPLETE.json").write_text(
    json.dumps(completion, indent=2), encoding="utf-8"
)
print(json.dumps(completion, indent=2))
